In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 43.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import pdfplumber
import pandas as pd
import re
import os
import glob
import csv

# 1. CONECTAR TU GOOGLE DRIVE
drive.mount('/content/drive')

# ==========================================
# TUS RUTAS
# ==========================================
carpeta_drive = "/content/drive/MyDrive/02_Semaforización/PL_SEMA_PDF"
# NUEVO ARCHIVO para no mezclar con el intento fallido
archivo_salida_csv = "/content/drive/MyDrive/Semaforos_Consolidados_V2.csv"

def procesar_sitraffic_por_texto():
    archivos_pdf = glob.glob(os.path.join(carpeta_drive, "*.pdf"))
    total_archivos = len(archivos_pdf)
    print(f"\nTotal de PDFs en la carpeta: {total_archivos}")

    archivos_procesados = set()
    if os.path.exists(archivo_salida_csv):
        try:
            df_previo = pd.read_csv(archivo_salida_csv, sep=';')
            archivos_procesados = set(df_previo['Archivo Origen'].unique())
            print(f"Se encontraron {len(archivos_procesados)} archivos ya listos en el V2. Se saltarán.")
        except: pass

    columnas = ["Archivo Origen", "ID Interseccion", "Direccion", "Plan de Senal", "Ciclo (s)", "Grupo de Senal (SG)", "TVI", "TVF", "TFD"]

    if not os.path.exists(archivo_salida_csv):
        with open(archivo_salida_csv, mode='w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow(columnas)

    archivos_pendientes = [f for f in archivos_pdf if os.path.basename(f) not in archivos_procesados]
    print(f"Archivos pendientes por procesar: {len(archivos_pendientes)}\n")
    print("-" * 50)

    for i, ruta_pdf in enumerate(archivos_pendientes, 1):
        nombre_archivo = os.path.basename(ruta_pdf)
        print(f"[{i}/{len(archivos_pendientes)}] Leyendo texto crudo de: {nombre_archivo}...")

        datos_pdf = []
        id_interseccion, direccion, plan_senal, ciclo = "N/A", "N/A", "N/A", "N/A"

        try:
            with pdfplumber.open(ruta_pdf) as pdf:
                for pagina in pdf.pages:
                    # Extraer el texto conservando la disposición aproximada
                    texto = pagina.extract_text(layout=True)
                    if not texto:
                        texto = pagina.extract_text() # Respaldo
                    if not texto: continue

                    # 1. ATRAPAR METADATOS
                    match_id = re.search(r'Externo[:\s]+(\d+)', texto, re.IGNORECASE)
                    if match_id: id_interseccion = match_id.group(1)

                    match_dir = re.search(r'Direcci[oó]n[:\s]+([^\n]+)', texto, re.IGNORECASE)
                    if match_dir: direccion = match_dir.group(1).strip()

                    match_ps = re.search(r'PS[_\s]+(\d+[a-zA-Z0-9_]*)', texto, re.IGNORECASE)
                    if match_ps: plan_senal = f"PS {match_ps.group(1)}"

                    match_ciclo = re.search(r'(Tiempo de Ciclo|TC=)\s*(\d+)', texto, re.IGNORECASE)
                    if match_ciclo: ciclo = match_ciclo.group(2)

                    # 2. ATRAPAR TIEMPOS (Línea por línea)
                    lineas = texto.split('\n')
                    en_zona_tiempos = False

                    for linea in lineas:
                        # Activar el "radar" de tiempos si vemos los encabezados
                        if 'TVI' in linea or 'TVF' in linea or 'TFD' in linea:
                            en_zona_tiempos = True
                            continue

                        # Apagar el "radar" si llegamos al final de esa sección
                        if 'Punto de conexi' in linea or 'SOP' in linea or 'ngos de punto' in linea:
                            en_zona_tiempos = False

                        if en_zona_tiempos:
                            # FÓRMULA MAGICA: Busca inicio con número (SG), espacio(s), y termina con 3 o 4 números (TVI, TVF, TFD)
                            match = re.search(r'^(\d+(?:\.\d+)?)\s+.*?(\d{1,3})\s+(\d{1,3})\s+(\d{1,3})(?:\s+\d{1,3})?\s*$', linea.strip())
                            if match:
                                sg = match.group(1)
                                tvi = match.group(2)
                                tvf = match.group(3)
                                tfd = match.group(4)
                                datos_pdf.append([nombre_archivo, id_interseccion, direccion, plan_senal, ciclo, sg, tvi, tvf, tfd])

        except Exception as e:
            print(f"  -> Error al leer el PDF: {e}")

        # Guardado Inmediato
        with open(archivo_salida_csv, mode='a', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f, delimiter=';')
            if datos_pdf:
                writer.writerows(datos_pdf)
            else:
                writer.writerow([nombre_archivo, "Sin datos legibles", "", "", "", "", "", "", ""])

    print("\n¡PROCESO FINALIZADO!")

procesar_sitraffic_por_texto()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Total de PDFs en la carpeta: 1376
Archivos pendientes por procesar: 1376

--------------------------------------------------
[1/1376] Leyendo texto crudo de: 1566.pdf...
[2/1376] Leyendo texto crudo de: 1567.pdf...
[3/1376] Leyendo texto crudo de: 1568.pdf...
[4/1376] Leyendo texto crudo de: 1570.pdf...
[5/1376] Leyendo texto crudo de: 1571.pdf...
[6/1376] Leyendo texto crudo de: 1575.pdf...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
df_prueba = pd.read_csv("/content/drive/MyDrive/Semaforos_RayosX.csv", sep=';')
df_prueba.head(30)


,Archivo Origen,ID Interseccion,Direccion,Plan de Senal,Ciclo (s),Grupo de Senal (SG),TVI,TVF,TFD
0,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,1,0.0,00,7
1,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,4,4.0,00,6
2,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,2,5.0,00,1
3,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,3,6.0,00,2
4,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,08,9.0,00,9
5,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,10,10.0,00,3
6,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,11,12.0,00,10
7,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,12,13.0,00,4
8,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,15,15.0,30,10
9,1566.pdf,1566,NÚMERO EXTERNO: 1566,PS 1,NaN,16,16.0,30,8


In [ ]:
from google.colab import drive
import pdfplumber
import pandas as pd
import re
import os
import glob
import csv

# 1. CONECTAR TU GOOGLE DRIVE
drive.mount('/content/drive')

carpeta_drive = "/content/drive/MyDrive/02_Semaforización/PL_SEMA_PDF"
# NUEVO ARCHIVO DE PRUEBA
archivo_salida_csv = "/content/drive/MyDrive/Semaforos_RayosX.csv"

def procesar_por_rayos_x():
    archivos_pdf = glob.glob(os.path.join(carpeta_drive, "*.pdf"))

    # ¡SOLO VAMOS A PROBAR CON 10 ARCHIVOS PARA NO PERDER TIEMPO!
    archivos_prueba = archivos_pdf[:10]
    print(f"\nIniciando prueba de Rayos X con {len(archivos_prueba)} archivos...\n" + "-"*50)

    columnas = ["Archivo Origen", "ID Interseccion", "Direccion", "Plan de Senal", "Ciclo (s)", "Grupo de Senal (SG)", "TVI", "TVF", "TFD"]

    # Crear archivo limpio
    with open(archivo_salida_csv, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(columnas)

    for i, ruta_pdf in enumerate(archivos_prueba, 1):
        nombre_archivo = os.path.basename(ruta_pdf)
        print(f"[{i}/{len(archivos_prueba)}] Escaneando píxeles de: {nombre_archivo}...")

        datos_pdf = []
        id_interseccion, direccion, plan_senal, ciclo = "N/A", "N/A", "N/A", "N/A"

        try:
            with pdfplumber.open(ruta_pdf) as pdf:
                for pagina in pdf.pages:
                    # 1. Metadatos generales (buscando en el texto normal)
                    texto = pagina.extract_text()
                    if texto:
                        match_id = re.search(r'Externo[:\s]+(\d+)', texto, re.IGNORECASE)
                        if match_id: id_interseccion = match_id.group(1)
                        match_dir = re.search(r'Direcci[oó]n[:\s]+([^\n]+)', texto, re.IGNORECASE)
                        if match_dir: direccion = match_dir.group(1).strip()
                        match_ps = re.search(r'PS[_\s]+(\d+[a-zA-Z0-9_]*)', texto, re.IGNORECASE)
                        if match_ps: plan_senal = f"PS {match_ps.group(1)}"
                        match_ciclo = re.search(r'(Tiempo de Ciclo|TC=)\s*(\d+)', texto, re.IGNORECASE)
                        if match_ciclo: ciclo = match_ciclo.group(2)

                    # 2. RAYOS X: LEER COORDENADAS EXACTAS DE CADA NUMERITO
                    palabras = pagina.extract_words()

                    # Agrupar numeritos que estén a la misma altura (tolerancia de 4 pixeles)
                    filas_visuales = {}
                    for p in palabras:
                        y = round(p['top'] / 4) * 4
                        if y not in filas_visuales: filas_visuales[y] = []
                        filas_visuales[y].append(p)

                    # Revisar cada "línea horizontal" que encontró el escaner
                    for y in sorted(filas_visuales.keys()):
                        fila = sorted(filas_visuales[y], key=lambda w: w['x0'])

                        # Buscar números a la IZQUIERDA (SG) y a la DERECHA (Tiempos)
                        # Ignoramos lo que esté en el medio (el gráfico de barras)
                        numeros_izq = [w['text'] for w in fila if re.match(r'^\d+(?:\.\d+)?$', w['text']) and w['x0'] < 300]
                        numeros_der = [w['text'] for w in fila if re.match(r'^\d+$', w['text']) and w['x0'] > 400]

                        # CRITERIO MÁGICO: Si hay un número a la izquierda, y al menos 3 a la derecha... ¡Es nuestra fila!
                        if numeros_izq and len(numeros_der) >= 3:
                            sg = numeros_izq[0]
                            tvi = numeros_der[0]
                            tvf = numeros_der[1]
                            tfd = numeros_der[2]

                            datos_pdf.append([nombre_archivo, id_interseccion, direccion, plan_senal, ciclo, sg, tvi, tvf, tfd])

        except Exception as e:
            print(f"  -> Error leyendo el archivo: {e}")

        # Guardar en Drive
        with open(archivo_salida_csv, mode='a', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f, delimiter=';')
            if datos_pdf:
                writer.writerows(datos_pdf)
            else:
                writer.writerow([nombre_archivo, "Sin datos geometricos", "", "", "", "", "", "", ""])

    print("\n¡PRUEBA FINALIZADA! Revisa el archivo Semaforos_RayosX.csv en tu Drive.")

# Ejecutar
procesar_por_rayos_x()

Mounted at /content/drive

Iniciando prueba de Rayos X con 10 archivos...
--------------------------------------------------
[1/10] Escaneando píxeles de: 1566.pdf...
[2/10] Escaneando píxeles de: 1567.pdf...
[3/10] Escaneando píxeles de: 1568.pdf...
[4/10] Escaneando píxeles de: 1570.pdf...
[5/10] Escaneando píxeles de: 1571.pdf...
[6/10] Escaneando píxeles de: 1575.pdf...
[7/10] Escaneando píxeles de: 1576.pdf...
[8/10] Escaneando píxeles de: 1580.pdf...
[9/10] Escaneando píxeles de: 1585.pdf...
[10/10] Escaneando píxeles de: 1586.pdf...

¡PRUEBA FINALIZADA! Revisa el archivo Semaforos_RayosX.csv en tu Drive.


In [ ]:
from google.colab import drive
import pdfplumber
import pandas as pd
import re
import os
import glob
import csv

# 1. CONECTAR TU GOOGLE DRIVE
drive.mount('/content/drive')

carpeta_drive = "/content/drive/MyDrive/02_Semaforización/PL_SEMA_PDF"
# NUEVO ARCHIVO DE PRUEBA V6
archivo_salida_csv = "/content/drive/MyDrive/Semaforos_RayosX_V6.csv"

def procesar_por_rayos_x_v6():
    archivos_pdf = glob.glob(os.path.join(carpeta_drive, "*.pdf"))

    # SEGUIMOS CON LA PRUEBA DE 10 ARCHIVOS
    archivos_prueba = archivos_pdf[:10]
    print(f"\nIniciando prueba V6 (Ajuste de alineación vertical) con {len(archivos_prueba)} archivos...\n" + "-"*50)

    columnas = ["Archivo Origen", "ID Interseccion", "Direccion", "Plan de Senal", "Ciclo (s)", "Grupo de Senal (SG)", "TVI", "TVF", "TFD"]

    with open(archivo_salida_csv, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(columnas)

    for i, ruta_pdf in enumerate(archivos_prueba, 1):
        nombre_archivo = os.path.basename(ruta_pdf)
        print(f"[{i}/{len(archivos_prueba)}] Escaneando: {nombre_archivo}...")

        datos_pdf = []
        id_interseccion, direccion, plan_senal, ciclo = "N/A", "N/A", "N/A", "N/A"

        try:
            with pdfplumber.open(ruta_pdf) as pdf:
                for pagina in pdf.pages:
                    texto = pagina.extract_text()
                    if texto:
                        match_id = re.search(r'Externo[:\s]+(\d+)', texto, re.IGNORECASE)
                        if match_id: id_interseccion = match_id.group(1)
                        match_dir = re.search(r'Direcci[oó]n[:\s]+([^\n]+)', texto, re.IGNORECASE)
                        if match_dir: direccion = match_dir.group(1).strip()
                        match_ps = re.search(r'PS[_\s]+(\d+[a-zA-Z0-9_]*)', texto, re.IGNORECASE)
                        if match_ps: plan_senal = f"PS {match_ps.group(1)}"
                        match_ciclo = re.search(r'(Tiempo de Ciclo|TC=)\s*(\d+)', texto, re.IGNORECASE)
                        if match_ciclo: ciclo = match_ciclo.group(2)

                    # 2. RAYOS X: LEER COORDENADAS CON MAYOR TOLERANCIA
                    palabras = pagina.extract_words()
                    filas_visuales = {}
                    for p in palabras:
                        # AJUSTE CLAVE: Usamos el centro vertical del número y damos 10 píxeles de tolerancia
                        centro_y = (p['top'] + p['bottom']) / 2
                        y = round(centro_y / 10) * 10

                        if y not in filas_visuales: filas_visuales[y] = []
                        filas_visuales[y].append(p)

                    for y in sorted(filas_visuales.keys()):
                        fila = sorted(filas_visuales[y], key=lambda w: w['x0'])

                        # Limpiamos textos como "SG 2" o "2." para quedarnos solo con el número
                        numeros_izq = []
                        for w in fila:
                            if w['x0'] < 300:
                                texto_limpio = re.sub(r'[^0-9\.]', '', w['text']).strip('.')
                                if texto_limpio:
                                    numeros_izq.append(texto_limpio)

                        numeros_der = [w['text'] for w in fila if re.match(r'^\d+$', w['text']) and w['x0'] > 400]

                        if numeros_izq and len(numeros_der) >= 3:
                            sg = numeros_izq[0]
                            tvi = numeros_der[0]
                            tvf = numeros_der[1]
                            tfd = numeros_der[2]
                            datos_pdf.append([nombre_archivo, id_interseccion, direccion, plan_senal, ciclo, sg, tvi, tvf, tfd])

        except Exception as e:
            pass # Silenciamos errores para no detener la prueba

        with open(archivo_salida_csv, mode='a', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f, delimiter=';')
            if datos_pdf:
                writer.writerows(datos_pdf)
            else:
                writer.writerow([nombre_archivo, "Sin datos geometricos", "", "", "", "", "", "", ""])

    print("\n¡PRUEBA V6 FINALIZADA! Revisa el archivo Semaforos_RayosX_V6.csv en tu Drive.")

procesar_por_rayos_x_v6()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Iniciando prueba V6 (Ajuste de alineación vertical) con 10 archivos...
--------------------------------------------------
[1/10] Escaneando: 1566.pdf...
[2/10] Escaneando: 1567.pdf...
[3/10] Escaneando: 1568.pdf...
[4/10] Escaneando: 1570.pdf...
[5/10] Escaneando: 1571.pdf...
[6/10] Escaneando: 1575.pdf...
[7/10] Escaneando: 1576.pdf...
[8/10] Escaneando: 1580.pdf...
[9/10] Escaneando: 1585.pdf...
[10/10] Escaneando: 1586.pdf...

¡PRUEBA V6 FINALIZADA! Revisa el archivo Semaforos_RayosX_V6.csv en tu Drive.


In [ ]:
from google.colab import drive
import pdfplumber
import pandas as pd
import re
import os
import glob
import csv
import unicodedata

# 1. CONECTAR TU GOOGLE DRIVE
drive.mount('/content/drive')

carpeta_drive = "/content/drive/MyDrive/02_Semaforización/PL_SEMA_PDF"
archivo_salida_csv = "/content/drive/MyDrive/Semaforos_Maestro_Claves.csv"

def normalizar_texto(texto):
    if not texto:
        return ""
    txt = unicodedata.normalize('NFD', texto)
    txt = ''.join(ch for ch in txt if unicodedata.category(ch) != 'Mn')
    return txt.lower()

def extraer_metadatos(texto, meta):
    if not texto:
        return

    if meta['Externo'] == "N/A":
        m = re.search(r'(?:numero\s+externo|n[uú]mero\s+externo|externo)\s*[:\-]?\s*(\d+)', texto, re.IGNORECASE)
        if m:
            meta['Externo'] = m.group(1)

    if meta['Direccion'] == "N/A":
        m = re.search(r'direcci[oó]n\s*[:\-]?\s*([^\n\r]+)', texto, re.IGNORECASE)
        if m:
            meta['Direccion'] = m.group(1).strip()

    if meta['Zona'] == "N/A":
        m = re.search(r'zona\s*[:\-]?\s*(\d+)', texto, re.IGNORECASE)
        if m:
            meta['Zona'] = m.group(1)

    if meta['Version'] == "N/A":
        m = re.search(r'versi[oó]n\s*[:\-]?\s*([^\n\r]+)', texto, re.IGNORECASE)
        if m:
            meta['Version'] = m.group(1).strip()

    if meta['Fecha'] == "N/A":
        m = re.search(r'fecha\s*[:\-]?\s*([^\n\r]+)', texto, re.IGNORECASE)
        if m:
            meta['Fecha'] = m.group(1).strip()

def metadatos_completos(meta):
    return all(v != "N/A" for v in meta.values())

def extraer_planes_desde_texto(texto):
    planes = []
    if not texto:
        return planes

    lineas = [l.strip() for l in texto.splitlines() if l.strip()]

    for i, linea in enumerate(lineas):
        m_ps = re.search(r'\bPS[_\s:-]*(\d+)\b', linea, re.IGNORECASE)
        if not m_ps:
            continue

        ps = m_ps.group(1)
        ciclo = None

        # Buscar ciclo en la misma linea y en las 3 siguientes
        ventana = lineas[i:i+4]
        for l in ventana:
            m_ciclo = re.search(r'(?:tiempo\s+de\s+ciclo|\bciclo\b|\bTC\b)\s*[:=]?\s*(\d+)\b', l, re.IGNORECASE)
            if m_ciclo:
                ciclo = m_ciclo.group(1)
                break

        planes.append((ps, ciclo if ciclo else "N/A"))

    # Quitar duplicados manteniendo orden
    vistos = set()
    unicos = []
    for p in planes:
        if p not in vistos:
            vistos.add(p)
            unicos.append(p)

    return unicos

def procesar_planes_clave():
    archivos_pdf = glob.glob(os.path.join(carpeta_drive, "*.pdf"))
    print(f"\nTotal de PDFs encontrados en la carpeta: {len(archivos_pdf)}")

    archivos_procesados = set()
    if os.path.exists(archivo_salida_csv):
        try:
            df_previo = pd.read_csv(archivo_salida_csv, sep=';')
            archivos_procesados = set(df_previo['Archivo Origen'].unique())
            print(f"Memoria activa: {len(archivos_procesados)} archivos ya procesados, se saltan.")
        except Exception:
            pass

    columnas = ["Archivo Origen", "Externo", "Direccion", "Zona", "Version", "Fecha", "PS", "Ciclo"]

    if not os.path.exists(archivo_salida_csv):
        with open(archivo_salida_csv, mode='w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow(columnas)

    pendientes = [f for f in archivos_pdf if os.path.basename(f) not in archivos_procesados]
    print(f"Archivos pendientes: {len(pendientes)}\n" + "-" * 60)

    for i, ruta_pdf in enumerate(pendientes, 1):
        nombre_archivo = os.path.basename(ruta_pdf)
        print(f"[{i}/{len(pendientes)}] Procesando: {nombre_archivo}")

        filas_salida = []
        meta = {
            "Externo": "N/A",
            "Direccion": "N/A",
            "Zona": "N/A",
            "Version": "N/A",
            "Fecha": "N/A",
        }

        planes_pdf = []

        try:
            with pdfplumber.open(ruta_pdf) as pdf:
                total_paginas = len(pdf.pages)
                paginas_texto = []
                for pagina in pdf.pages:
                    txt = pagina.extract_text(layout=True) or pagina.extract_text() or ""
                    paginas_texto.append(txt)

                # 1) METADATOS UNICOS
                # Prioridad A: paginas con titulo "Esquema"
                idx_esquema = [idx for idx, t in enumerate(paginas_texto) if 'esquema' in normalizar_texto(t)]
                for idx in idx_esquema:
                    extraer_metadatos(paginas_texto[idx], meta)
                    if metadatos_completos(meta):
                        break

                # Prioridad B: pagina 3 (indice 2)
                if not metadatos_completos(meta) and total_paginas >= 3:
                    extraer_metadatos(paginas_texto[2], meta)

                # Prioridad C: seguir pagina a pagina hasta completar
                if not metadatos_completos(meta):
                    for txt in paginas_texto:
                        extraer_metadatos(txt, meta)
                        if metadatos_completos(meta):
                            break

                # 2) PLANES: solo paginas con titulos clave
                titulos_planes = (
                    'programas de senal',
                    'programas de regulacion semaforica',
                )

                for txt in paginas_texto:
                    tnorm = normalizar_texto(txt)
                    if any(tit in tnorm for tit in titulos_planes):
                        planes_pdf.extend(extraer_planes_desde_texto(txt))

            # Deduplicar planes por PDF
            vistos = set()
            planes_unicos = []
            for plan in planes_pdf:
                if plan not in vistos:
                    vistos.add(plan)
                    planes_unicos.append(plan)

            if planes_unicos:
                for ps, ciclo in planes_unicos:
                    filas_salida.append([
                        nombre_archivo,
                        meta['Externo'],
                        meta['Direccion'],
                        meta['Zona'],
                        meta['Version'],
                        meta['Fecha'],
                        ps,
                        ciclo,
                    ])
            else:
                filas_salida.append([
                    nombre_archivo,
                    meta['Externo'],
                    meta['Direccion'],
                    meta['Zona'],
                    meta['Version'],
                    meta['Fecha'],
                    "N/A",
                    "N/A",
                ])

        except Exception as e:
            filas_salida.append([nombre_archivo, "ERROR", str(e), "", "", "", "", ""])

        with open(archivo_salida_csv, mode='a', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerows(filas_salida)

    print("\nProceso finalizado. Archivo generado: Semaforos_Maestro_Claves.csv")

# Ejecutar
procesar_planes_clave()

